# Building the ED simulation in stages

This is the model from the Week 04 session, built one idea at a time, as in the session plan. Run each stage and answer
its question before you move on. Each stage is complete and runs on its own; later stages reuse the Stage 3 function.

1. One queue, every patient alike
2. Add triage
3. Add patient types
4. Priority
5. Pre-emption (interruptions)
6. Experiments with 2–5 doctors

*Made for this site from the staff notes `5 Source Files/Staff Notes/Building the simulation live.md`, whose code is
written as fragments. The finished model, with costs and confidence intervals, is `2 Emergency department simulation (full model).ipynb`.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install simpy

In [2]:
import simpy
import random
import numpy as np

SIM_TIME = 12 * 60          # one 12-hour shift, in minutes
ARRIVAL_RATE = 4 / 60       # 4 patients per hour

## Stage 1: basic ED

Patients arrive at random (4 per hour) and wait for one of the doctors (25-minute mean treatment).

**Ask:** What assumptions are unrealistic?

In [3]:
# Stage 1: one queue, every patient alike
def run_stage1(n_doctors=2, seed=1):
    random.seed(seed)
    waits = []

    def patient(env, doctors):
        arrival = env.now
        with doctors.request() as req:
            yield req
            waits.append(env.now - arrival)
            service = random.expovariate(1 / 25)
            yield env.timeout(service)

    def arrivals(env, doctors):
        while True:
            yield env.timeout(random.expovariate(ARRIVAL_RATE))
            env.process(patient(env, doctors))

    env = simpy.Environment()
    doctors = simpy.Resource(env, capacity=n_doctors)
    env.process(arrivals(env, doctors))
    env.run(until=SIM_TIME)
    print(f"Stage 1, {n_doctors} doctors: {len(waits)} patients seen, average wait {np.mean(waits):.1f} min")

run_stage1()

Stage 1, 2 doctors: 49 patients seen, average wait 21.4 min


## Stage 2: add triage

The ED is not one queue: a triage nurse sees every patient first (4–8 minutes).

**Ask:** Where is the congestion now?

In [4]:
# Stage 2: add a triage nurse in front of the doctors
def run_stage2(n_doctors=2, seed=1):
    random.seed(seed)
    tri_waits, doc_waits, tri_busy = [], [], [0.0]

    def patient(env, triage, doctors):
        arrival = env.now
        with triage.request() as req:
            yield req
            tri_waits.append(env.now - arrival)
            t = random.uniform(4, 8)
            tri_busy[0] += t
            yield env.timeout(t)
        after_triage = env.now
        with doctors.request() as req:
            yield req
            doc_waits.append(env.now - after_triage)
            yield env.timeout(random.expovariate(1 / 25))

    def arrivals(env, triage, doctors):
        while True:
            yield env.timeout(random.expovariate(ARRIVAL_RATE))
            env.process(patient(env, triage, doctors))

    env = simpy.Environment()
    triage = simpy.Resource(env, capacity=1)
    doctors = simpy.Resource(env, capacity=n_doctors)
    env.process(arrivals(env, triage, doctors))
    env.run(until=SIM_TIME)
    print(f"Stage 2, {n_doctors} doctors: triage wait {np.mean(tri_waits):.1f} min, doctor wait {np.mean(doc_waits):.1f} min, "
          f"triage busy {tri_busy[0] / SIM_TIME:.0%} of the shift")

run_stage2()

Stage 2, 2 doctors: triage wait 2.7 min, doctor wait 49.0 min, triage busy 43% of the shift


## Stage 3: add patient types

10% critical (40 min), 30% urgent (30 min), 60% standard (20 min), still in one first-come-first-served queue.

**Ask:** Should all patients wait in the same queue?

In [5]:
# Stage 3: three patient types with different treatment times
SERVICE = {"critical": 40, "urgent": 30, "standard": 20}   # mean minutes

def sample_type():
    r = random.random()
    if r < 0.1:
        return "critical"
    elif r < 0.4:
        return "urgent"
    return "standard"

def run_stage3(n_doctors=2, seed=1, resource="plain", label="Stage 3"):
    """resource: "plain" (one FIFO queue), "priority" (Stage 4) or "preemptive" (Stage 5)."""
    random.seed(seed)
    PRIORITY = {"critical": 0, "urgent": 1, "standard": 2}
    waits = {t: [] for t in SERVICE}

    def patient(env, triage, doctors):
        with triage.request() as req:
            yield req
            yield env.timeout(random.uniform(4, 8))
        after_triage = env.now
        ptype = sample_type()
        service = random.expovariate(1 / SERVICE[ptype])
        if resource == "plain":
            with doctors.request() as req:
                yield req
                waits[ptype].append(env.now - after_triage)
                yield env.timeout(service)
        elif resource == "priority":
            with doctors.request(priority=PRIORITY[ptype]) as req:
                yield req
                waits[ptype].append(env.now - after_triage)
                yield env.timeout(service)
        else:   # Stage 5: pre-emption with resume, as in the session notes
            remaining, seen = service, False
            while remaining > 0:
                try:
                    with doctors.request(priority=PRIORITY[ptype], preempt=True) as req:
                        yield req
                        if not seen:
                            waits[ptype].append(env.now - after_triage)
                            seen = True
                        start = env.now
                        yield env.timeout(remaining)
                        remaining = 0
                except simpy.Interrupt:
                    remaining -= env.now - start      # queue again for the rest of the treatment

    def arrivals(env, triage, doctors):
        while True:
            yield env.timeout(random.expovariate(ARRIVAL_RATE))
            env.process(patient(env, triage, doctors))

    env = simpy.Environment()
    triage = simpy.Resource(env, capacity=1)
    doctors = {"plain": simpy.Resource, "priority": simpy.PriorityResource,
               "preemptive": simpy.PreemptiveResource}[resource](env, capacity=n_doctors)
    env.process(arrivals(env, triage, doctors))
    env.run(until=SIM_TIME)
    summary = ", ".join(f"{t} {np.mean(w):.1f} min" for t, w in waits.items() if w)
    print(f"{label}, {n_doctors} doctors, average wait for a doctor: {summary}")
    return {t: np.mean(w) for t, w in waits.items() if w}

_ = run_stage3()

Stage 3, 2 doctors, average wait for a doctor: critical 9.1 min, urgent 17.4 min, standard 21.3 min


## Stage 4: priority queue

`PriorityResource`: a free doctor takes the most urgent patient waiting.

**Ask:** Is priority enough? (An urgent patient can still wait behind a standard one already being treated.)

In [6]:
# Stage 4: the doctors see the most urgent waiting patient first (no interruptions yet)
_ = run_stage3(resource="priority", label="Stage 4")

Stage 4, 2 doctors, average wait for a doctor: critical 1.6 min, urgent 11.9 min, standard 25.0 min


## Stage 5: pre-emption

`PreemptiveResource`: a more urgent patient interrupts a doctor. The interrupted patient queues again for the rest of their treatment.

**Ask:** What policy decisions exist?

In [7]:
# Stage 5: a critical patient can interrupt a doctor who is treating a less urgent patient
_ = run_stage3(resource="preemptive", label="Stage 5")

Stage 5, 2 doctors, average wait for a doctor: critical 0.0 min, urgent 6.8 min, standard 20.9 min


## Stage 6: experiments

One shift is a single random sample, so each staffing level is averaged over 20 shifts.

**Ask:** How many doctors should the ED employ? What else would you need to know (costs, targets)?

In [8]:
import contextlib, io

# Stage 6: experiments. Average over 20 shifts (seeds) for each number of doctors
for d in [2, 3, 4, 5]:
    runs = []
    for seed in range(20):
        with contextlib.redirect_stdout(io.StringIO()):     # hide the per-run line
            runs.append(run_stage3(n_doctors=d, seed=seed, resource="preemptive"))
    mean = {t: np.mean([r[t] for r in runs if t in r]) for t in SERVICE}
    print(f"{d} doctors: " + ", ".join(f"{t} {m:.1f} min" for t, m in mean.items()))

2 doctors: critical 0.1 min, urgent 3.1 min, standard 32.1 min
3 doctors: critical 0.0 min, urgent 0.4 min, standard 4.4 min
4 doctors: critical 0.0 min, urgent 0.0 min, standard 0.6 min
5 doctors: critical 0.0 min, urgent 0.0 min, standard 0.0 min
